In [1]:
import pandas as pd
df = pd.read_csv('data/SGJobData.csv')


In [2]:
df_clean = df.dropna(subset=["metadata_jobPostId"]).copy()

# Remove columns with no useful data
df_clean = df_clean.drop(columns=["occupationId"])

print("Original rows:", len(df))
print("Cleaned rows:", len(df_clean))

Original rows: 1048585
Cleaned rows: 1044597


In [3]:
# Check for invalid salary ranges
invalid_salary = df_clean[
    df_clean["salary_minimum"] > df_clean["salary_maximum"]
]

# Check for negative values
invalid_values = df_clean[
    (df_clean["salary_minimum"] < 0) |
    (df_clean["salary_maximum"] < 0) |
    (df_clean["minimumYearsExperience"] < 0) |
    (df_clean["numberOfVacancies"] < 0)
]

print("Invalid salary ranges:", len(invalid_salary))
print("Negative values:", len(invalid_values))

Invalid salary ranges: 0
Negative values: 0


In [4]:
Q1 = df_clean["average_salary"].quantile(0.25)
Q3 = df_clean["average_salary"].quantile(0.75)

IQR = Q3 - Q1

upper_limit = Q3 + (1.5 * IQR)
lower_limit = Q1 - (1.5 * IQR)

# Flag potential outliers
df_clean["salary_outlier"] = (
    (df_clean["average_salary"] < lower_limit) |
    (df_clean["average_salary"] > upper_limit)
)

print("Potential salary outliers:",
      df_clean["salary_outlier"].sum())

Potential salary outliers: 70837


In [14]:
# Original dataset


print("Original mean:",
      df_clean["average_salary"].mean().round(1))

print("Original median:",
      df_clean["average_salary"].median())


# Dataset without flagged salary outliers (outside of Q1 and Q3)
df_no_outliers = df_clean[
    ~df_clean["salary_outlier"]
]

print("Q1 and Q3", Q1, Q3)

print("Cleaned mean:",
      df_no_outliers["average_salary"].mean().round(1))

print("Cleaned median:",
      df_no_outliers["average_salary"].median())

print("Mean change %%!:",
      ((df_clean["average_salary"].mean() - df_no_outliers["average_salary"].mean()) / df_clean["average_salary"].mean() * 100).round(1))

print("Median change %%!:",
      ((df_clean["average_salary"].median() - df_no_outliers["average_salary"].median()) / df_clean["average_salary"].median() * 100))


Original mean: 4787.7
Original median: 3800.0
Q1 and Q3 2900.0 5500.0
Cleaned mean: 4069.5
Cleaned median: 3750.0
Mean change %%!: 15.0
Median change %%!: 1.3157894736842104
